# Next-Concept Modeling, hands-on

Toy replay of the NCM paper (arXiv:2610.14162, Stanford Hazy Research + Together AI, Oct 9 2026):
residual quantization error falling level by level, the step-count arithmetic (levels K vs tokens L),
and the paper's headline numbers recomputed from its own tables.

In [ ]:
import numpy as np

# Toy residual quantization: a 4-dim 'meaning vector', quantized on grids
# that get 4x finer each level -- the same mechanics as NCM's residual VQ-VAE,
# minus the learned codebooks. Each level quantizes the residual of the last.
z = np.array([3.7, -1.2, 0.6, 2.9])
steps = [1.0, 0.25, 0.0625, 0.015625]  # grid spacing per level
recon = np.zeros(4)
errors = []
for k, step in enumerate(steps):
    residual = z - recon
    recon = recon + np.round(residual / step) * step
    errors.append(float(np.linalg.norm(z - recon)))
print("residual error by level:", [round(e, 4) for e in errors])
assert all(errors[i] >= errors[i + 1] for i in range(3)), "error must not grow"
assert errors[-1] < 0.35 * errors[0], "final error should be well under a third of level-1 error"
print("OK: each level corrects what the last one left out")

In [ ]:
# Step-count arithmetic: generating a 512-token passage.
L = 512
dlm_steps = 128          # illustrative full-sequence denoising passes
K = 12                   # NCM levels (Sudoku run in the paper used 12 NFE)
print(f"autoregressive steps : {L}")
print(f"diffusion steps      : {dlm_steps}")
print(f"NCM steps            : {K}")
print(f"NCM vs AR step ratio : {L / K:.1f}x fewer sequential steps")
print(f"NCM vs DLM step ratio: {dlm_steps / K:.1f}x fewer sequential steps")
assert L / K > 40 and dlm_steps / K > 10
print("OK: NCM's sequential path is levels, not tokens")

In [ ]:
# Paper numbers, recomputed (abstract + intro + Table 3).
speedups = {"LM1B": 21, "OpenWebText": 10, "XSum": 31}
for task, s in speedups.items():
    print(f"{task}: {s}x faster than strongest diffusion baseline")
assert min(speedups.values()) == 10 and max(speedups.values()) == 31

# Sudoku accuracy by difficulty: NCM (25M, 12 NFE) vs best diffusion baseline (29M, 16 NFE)
ncm  = {"easy": 99.5, "medium": 96.5, "hard": 86.1}
base = {"easy": 97.7, "medium": 92.8, "hard": 81.3}
for d in ncm:
    print(f"Sudoku {d}: NCM {ncm[d]} vs baseline {base[d]} -> +{ncm[d] - base[d]:.1f} pp")
assert round(ncm["hard"] - base["hard"], 1) == 4.8
assert (ncm["hard"] - base["hard"]) > (ncm["easy"] - base["easy"]), "gap widens with difficulty"
print(f"NFE: 12 vs 16 -> {16 / 12:.2f}x fewer function evaluations")
print(f"Training tokens: 4-5x fewer (paper Fig. 7); MBR decoding: +7% ROUGE-L at equal time budget")
print("OK: all headline numbers check out")